# 03 - Feature Engineering

**Leak-safe rule:** every feature at time *t* uses only information at or before *t*; the target is benzene at *t+1*. Each rolling statistic is computed on a `shift(1)`-ed series so a row can never see its own value.

- **Lags** 1/2/3/24 h - short momentum + the 24 h daily cycle
- **Rolling mean & std** over 3/6/24 h - local level + local volatility (the std exposes hardware instability)
- **Cyclical encodings** of hour / day-of-week / month via sin & cos
- **Current-hour readings**, including the latest benzene value (the same information the persistence baseline uses).

In [1]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))
import warnings; warnings.filterwarnings('ignore')
import pandas as pd, numpy as np
from IPython.display import Image, display
from src import config as C, features as fe
cleaned = pd.read_csv(C.DATA_PROCESSED, index_col=0, parse_dates=True)
feat = fe.build_features(cleaned)
print('feature matrix:', feat.shape, '|', feat.shape[1]-1, 'features')
feat.iloc[:3, :8]

feature matrix: (8853, 106) | 105 features


,PT08.S1(CO),PT08.S2(NMHC),PT08.S3(NOx),PT08.S4(NO2),PT08.S5(O3),T,RH,AH
2004-03-11 18:00:00,1581.0,1319.0,799.0,2083.0,1409.0,10.3,64.2,0.8065
2004-03-11 19:00:00,1776.0,1488.0,702.0,2333.0,1531.5,9.7,69.3,0.8319
2004-03-11 20:00:00,1640.0,1404.0,743.0,2191.0,1654.0,9.6,67.8,0.8133


### Chronological split - never shuffled
Train = earliest 70%, validation = next 15% (for early stopping), test = most recent 15%. Shuffling would leak future into past.

In [2]:
(Xtr,ytr),(Xva,yva),(Xte,yte) = fe.time_split(feat)
print(f'train={len(Xtr)}  val={len(Xva)}  test={len(Xte)}')
print('train ends:', Xtr.index.max())
print('test  starts:', Xte.index.min())

train=6199  val=1327  test=1327
train ends: 2004-12-03 14:00:00
test  starts: 2005-02-04 08:00:00
